# Audio Classification Pipeline — 10 Classes (ML + DL)

End-to-end Colab notebook: dataset loading from Google Drive → stratified
70/15/15 split → augmentation (train only) → acoustic feature extraction →
training 3 models (SVM, XGBoost, CNN on mel-spectrograms) → evaluation
(accuracy/precision/recall/F1/macro-F1/confusion matrix/class-wise/critical-event
recall/noise robustness) → overfitting checks → inference with per-class
confidence scores.

**Before running:** upload `audio_dataset.zip` to your Google Drive (root, or
edit `ZIP_PATH` below). The zip is expected to contain 10 subfolders (one per
class), each with ~300 audio files, e.g.:

```
audio_dataset/
  class_1/*.wav
  class_2/*.wav
  ...
  class_10/*.wav
```

If your folder names differ, nothing needs to change — the pipeline reads
class names directly from the subfolder names.


## Step 0 — Setup & Installs

In [ ]:
!pip install -q librosa==0.10.2.post1 soundfile audiomentations xgboost scikit-learn \
    tensorflow matplotlib seaborn pandas numpy tqdm


In [ ]:
import os, glob, shutil, zipfile, random, json, math, time, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import librosa
import librosa.display
import soundfile as sf
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                              f1_score, confusion_matrix, classification_report)

import xgboost as xgb
import joblib

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, regularizers

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

print("TF:", tf.__version__, "| GPU available:", tf.config.list_physical_devices('GPU'))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ---- EDIT THESE IF NEEDED ----
ZIP_PATH    = "/content/drive/MyDrive/audio_dataset.zip"   # path to zip in Drive
EXTRACT_DIR = "/content/audio_dataset"                     # local extraction target
WORK_DIR    = "/content/work"                               # scratch space (augmented audio, caches)
MODEL_DIR   = "/content/drive/MyDrive/audio_classification_models"  # where trained models are saved
SR          = 22050      # sample rate for all loading/processing
DURATION    = 4.0        # seconds — clips are padded/truncated to this length
N_MFCC      = 40
CRITICAL_CLASSES = []     # e.g. ["siren", "alarm"] — fill in with your safety-critical class names
# --------------------------------

os.makedirs(EXTRACT_DIR, exist_ok=True)
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

if not os.path.isdir(EXTRACT_DIR) or len(os.listdir(EXTRACT_DIR)) == 0:
    print("Extracting zip...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_DIR)
    print("Done.")
else:
    print("Already extracted.")


## Step 1 — Discover Dataset & Build File Index

In [ ]:
def find_class_root(extract_dir):
    """Handle the case where the zip has one nested top-level folder."""
    entries = [e for e in os.listdir(extract_dir) if not e.startswith('.')]
    if len(entries) == 1 and os.path.isdir(os.path.join(extract_dir, entries[0])):
        return os.path.join(extract_dir, entries[0])
    return extract_dir

DATA_ROOT = find_class_root(EXTRACT_DIR)
AUDIO_EXTS = (".wav", ".mp3", ".flac", ".ogg", ".m4a")

classes = sorted([d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d))])
assert len(classes) > 0, f"No class subfolders found under {DATA_ROOT}"
print(f"Found {len(classes)} classes:", classes)

records = []
for c in classes:
    files = [f for f in glob.glob(os.path.join(DATA_ROOT, c, "*")) if f.lower().endswith(AUDIO_EXTS)]
    for f in files:
        records.append({"filepath": f, "label": c})

df = pd.DataFrame(records)
print("Total audio files:", len(df))
print(df.groupby("label").size())


In [ ]:
# Sanity-check class balance
counts = df.groupby("label").size()
plt.figure(figsize=(9,4))
sns.barplot(x=counts.index, y=counts.values)
plt.xticks(rotation=45, ha="right")
plt.ylabel("file count")
plt.title("Files per class")
plt.tight_layout()
plt.show()


## Step 2 — Stratified 70 / 15 / 15 Split

Split is done on file paths only (no audio loaded yet) using two stratified
`train_test_split` calls so every class keeps the same proportion in each
split.

In [ ]:
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df["label"], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["label"], random_state=SEED
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print("\nTrain class balance:\n", train_df.label.value_counts())
print("\nVal class balance:\n", val_df.label.value_counts())
print("\nTest class balance:\n", test_df.label.value_counts())

label_encoder = LabelEncoder().fit(classes)
NUM_CLASSES = len(classes)


## Step 3 — Audio Loading Utility

In [ ]:
def load_audio(path, sr=SR, duration=DURATION):
    """Load, resample, mono-mix, and pad/truncate to a fixed length."""
    y, _ = librosa.load(path, sr=sr, mono=True)
    target_len = int(sr * duration)
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    else:
        y = y[:target_len]
    return y


## Step 4 — Data Augmentation (TRAINING SET ONLY)

Implements: background-noise addition, time shifting, pitch shifting, time
stretching, volume adjustment, limited reverberation, distance simulation,
and recording-device simulation (via band-limiting / mild distortion + EQ
coloring, a practical proxy for cheap-mic frequency response).

Augmentation is applied **only** to the training split — validation and test
must stay clean so metrics reflect real-world performance and don't leak
augmented variants of the same source clip across splits.

In [ ]:
def add_background_noise(y, snr_db=None):
    snr_db = snr_db if snr_db is not None else random.uniform(5, 20)
    noise = np.random.randn(len(y))
    sig_power = np.mean(y ** 2) + 1e-9
    noise_power = np.mean(noise ** 2) + 1e-9
    factor = np.sqrt(sig_power / (noise_power * (10 ** (snr_db / 10))))
    return y + factor * noise

def time_shift(y, max_frac=0.2):
    shift = int(len(y) * random.uniform(-max_frac, max_frac))
    return np.roll(y, shift)

def pitch_shift(y, sr=SR, max_steps=3):
    steps = random.uniform(-max_steps, max_steps)
    return librosa.effects.pitch_shift(y, sr=sr, n_steps=steps)

def time_stretch(y, rate_range=(0.85, 1.15)):
    rate = random.uniform(*rate_range)
    y_s = librosa.effects.time_stretch(y, rate=rate)
    # restore original fixed length
    target_len = len(y)
    if len(y_s) < target_len:
        y_s = np.pad(y_s, (0, target_len - len(y_s)))
    else:
        y_s = y_s[:target_len]
    return y_s

def volume_adjust(y, gain_db_range=(-6, 6)):
    gain_db = random.uniform(*gain_db_range)
    return y * (10 ** (gain_db / 20))

def limited_reverb(y, sr=SR, decay=0.3, n_reflections=3):
    """Cheap synthetic reverb: sum of decaying delayed copies."""
    out = y.copy()
    for i in range(1, n_reflections + 1):
        delay = int(sr * 0.02 * i)
        atten = decay ** i
        delayed = np.pad(y, (delay, 0))[:len(y)]
        out = out + atten * delayed
    return out / (1 + sum(decay ** i for i in range(1, n_reflections + 1)))

def distance_simulation(y, atten_db_range=(3, 12)):
    """Approximate a more distant source: attenuate + mild low-pass (high-freq loss)."""
    atten_db = random.uniform(*atten_db_range)
    y_att = y * (10 ** (-atten_db / 20))
    # simple one-pole low-pass
    alpha = random.uniform(0.15, 0.35)
    y_lp = np.zeros_like(y_att)
    prev = 0.0
    for i in range(len(y_att)):
        prev = alpha * y_att[i] + (1 - alpha) * prev
        y_lp[i] = prev
    return y_lp

def device_simulation(y, sr=SR):
    """Approximate a cheap mic/recorder: band-limit + slight nonlinear clipping."""
    # band-limit via STFT masking (~150 Hz - 7000 Hz)
    S = librosa.stft(y)
    freqs = librosa.fft_frequencies(sr=sr)
    mask = (freqs >= 150) & (freqs <= 7000)
    S[~mask, :] = S[~mask, :] * 0.15
    y_bl = librosa.istft(S, length=len(y))
    # mild soft clipping for device coloration
    drive = random.uniform(1.0, 1.6)
    y_out = np.tanh(y_bl * drive) / np.tanh(drive)
    return y_out

AUG_FUNCS = [add_background_noise, time_shift, pitch_shift, time_stretch,
             volume_adjust, limited_reverb, distance_simulation, device_simulation]

def augment_clip(y, n_ops=None):
    """Apply a random subset (1-3) of augmentations, in random order."""
    n_ops = n_ops or random.randint(1, 3)
    ops = random.sample(AUG_FUNCS, n_ops)
    out = y.copy()
    for op in ops:
        try:
            out = op(out)
        except Exception:
            pass  # skip an op if it fails on a pathological clip
    # guard against clipping/NaNs
    out = np.nan_to_num(out)
    peak = np.max(np.abs(out)) + 1e-9
    if peak > 1.0:
        out = out / peak
    return out


In [ ]:
AUGS_PER_CLIP = 2   # how many augmented copies to generate per original training clip
AUG_DIR = os.path.join(WORK_DIR, "augmented")
os.makedirs(AUG_DIR, exist_ok=True)
AUG_INDEX_PATH = os.path.join(WORK_DIR, "aug_index.csv")

# aug_df is ALWAYS defined below (even if empty), and train_full_df is ALWAYS
# built at the end of this cell — a failure on any single bad/corrupt file is
# caught and skipped instead of aborting the whole cell (which is what left
# train_full_df undefined before).
aug_records = []
if not os.path.exists(AUG_INDEX_PATH):
    n_failed = 0
    for _, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Augmenting train set"):
        try:
            y = load_audio(row.filepath)
        except Exception as e:
            n_failed += 1
            print(f"  [skip] could not load {row.filepath}: {e}")
            continue
        for k in range(AUGS_PER_CLIP):
            try:
                y_aug = augment_clip(y)
                out_name = f"{os.path.splitext(os.path.basename(row.filepath))[0]}_aug{k}.wav"
                out_path = os.path.join(AUG_DIR, row.label, out_name)
                os.makedirs(os.path.dirname(out_path), exist_ok=True)
                sf.write(out_path, y_aug, SR)
                aug_records.append({"filepath": out_path, "label": row.label})
            except Exception as e:
                n_failed += 1
                print(f"  [skip] augmentation failed for {row.filepath} (op {k}): {e}")
    if n_failed:
        print(f"Skipped {n_failed} file/augmentation operations due to errors (see above).")
    aug_df = pd.DataFrame(aug_records, columns=["filepath", "label"])
    aug_df.to_csv(AUG_INDEX_PATH, index=False)
else:
    aug_df = pd.read_csv(AUG_INDEX_PATH)

print("Augmented clips generated:", len(aug_df))

# Full training set = original + augmented (validation/test remain untouched, clean).
# This line always runs — aug_df is guaranteed to exist (possibly empty) above.
train_full_df = pd.concat([train_df, aug_df], ignore_index=True)
assert len(train_full_df) > 0, "train_full_df ended up empty — check the errors printed above."
print("Training set size after augmentation:", len(train_full_df))
print(train_full_df.label.value_counts())


## Step 5 — Acoustic Feature Extraction

Extracts MFCCs, mel spectrogram, chroma, zero-crossing rate, RMS energy,
spectral centroid, spectral bandwidth, spectral roll-off, onset strength,
and tempo. Two representations are produced:

- **Tabular features** (mean/std pooled over time) → for SVM / XGBoost
- **Mel-spectrogram image** (fixed-size 2-D array) → for the CNN

In [ ]:
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128

def extract_tabular_features(y, sr=SR):
    feats = {}

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH)
    for i in range(N_MFCC):
        feats[f"mfcc{i}_mean"] = np.mean(mfcc[i]); feats[f"mfcc{i}_std"] = np.std(mfcc[i])

    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS)
    mel_db = librosa.power_to_db(mel)
    feats["mel_mean"] = np.mean(mel_db); feats["mel_std"] = np.std(mel_db)

    chroma = librosa.feature.chroma_stft(y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH)
    for i in range(chroma.shape[0]):
        feats[f"chroma{i}_mean"] = np.mean(chroma[i])

    zcr = librosa.feature.zero_crossing_rate(y)
    feats["zcr_mean"] = np.mean(zcr); feats["zcr_std"] = np.std(zcr)

    rms = librosa.feature.rms(y=y)
    feats["rms_mean"] = np.mean(rms); feats["rms_std"] = np.std(rms)

    cent = librosa.feature.spectral_centroid(y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH)
    feats["centroid_mean"] = np.mean(cent); feats["centroid_std"] = np.std(cent)

    bw = librosa.feature.spectral_bandwidth(y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH)
    feats["bandwidth_mean"] = np.mean(bw); feats["bandwidth_std"] = np.std(bw)

    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH)
    feats["rolloff_mean"] = np.mean(rolloff); feats["rolloff_std"] = np.std(rolloff)

    onset_env = librosa.onset.onset_strength(y=y, sr=sr, hop_length=HOP_LENGTH)
    feats["onset_strength_mean"] = np.mean(onset_env); feats["onset_strength_std"] = np.std(onset_env)

    try:
        tempo, _ = librosa.beat.beat_track(y=y, sr=sr, hop_length=HOP_LENGTH)
        feats["tempo"] = float(tempo) if np.isscalar(tempo) else float(tempo[0])
    except Exception:
        feats["tempo"] = 0.0

    return feats

FIXED_MEL_FRAMES = int(np.ceil(SR * DURATION / HOP_LENGTH))

def extract_mel_image(y, sr=SR):
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS)
    mel_db = librosa.power_to_db(mel)
    # pad/truncate time axis to a fixed width
    if mel_db.shape[1] < FIXED_MEL_FRAMES:
        mel_db = np.pad(mel_db, ((0,0),(0, FIXED_MEL_FRAMES - mel_db.shape[1])))
    else:
        mel_db = mel_db[:, :FIXED_MEL_FRAMES]
    # per-clip normalization
    mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-9)
    return mel_db.astype(np.float32)


In [ ]:
def build_feature_table(subset_df, cache_path):
    if os.path.exists(cache_path):
        return pd.read_pickle(cache_path)
    rows = []
    for _, row in tqdm(subset_df.iterrows(), total=len(subset_df), desc=f"Features -> {os.path.basename(cache_path)}"):
        y = load_audio(row.filepath)
        f = extract_tabular_features(y)
        f["label"] = row.label
        rows.append(f)
    out = pd.DataFrame(rows)
    out.to_pickle(cache_path)
    return out

feat_train = build_feature_table(train_full_df, os.path.join(WORK_DIR, "feat_train.pkl"))
feat_val   = build_feature_table(val_df,        os.path.join(WORK_DIR, "feat_val.pkl"))
feat_test  = build_feature_table(test_df,       os.path.join(WORK_DIR, "feat_test.pkl"))

feature_cols = [c for c in feat_train.columns if c != "label"]
print("Tabular feature count:", len(feature_cols))


## Step 6 — Prepare Tabular Matrices (SVM / XGBoost)

In [ ]:
scaler = StandardScaler().fit(feat_train[feature_cols])

X_train = scaler.transform(feat_train[feature_cols]); y_train = label_encoder.transform(feat_train.label)
X_val   = scaler.transform(feat_val[feature_cols]);   y_val   = label_encoder.transform(feat_val.label)
X_test  = scaler.transform(feat_test[feature_cols]);  y_test  = label_encoder.transform(feat_test.label)

print(X_train.shape, X_val.shape, X_test.shape)


## Step 7a — Model 1: Support Vector Machine (with cross-validated grid search)

`GridSearchCV` with stratified k-fold cross-validation selects `C` / `gamma`
to avoid overfitting to a single split; scoring is macro-F1 to weight all
10 classes equally.

In [ ]:
svm_param_grid = {
    "C": [1, 5, 10, 20],
    "gamma": ["scale", 0.01, 0.001],
    "kernel": ["rbf"],
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

svm_search = GridSearchCV(
    SVC(probability=True, class_weight="balanced", random_state=SEED),
    svm_param_grid, scoring="f1_macro", cv=cv, n_jobs=-1, verbose=1
)
svm_search.fit(X_train, y_train)
svm_model = svm_search.best_estimator_
print("Best SVM params:", svm_search.best_params_)
print("Best CV macro-F1:", svm_search.best_score_)

# Save immediately so this model is safe on Drive even if a later cell fails
joblib.dump(svm_model, os.path.join(MODEL_DIR, "svm_model.joblib"))
with open(os.path.join(MODEL_DIR, "svm_best_params.json"), "w") as f:
    json.dump(svm_search.best_params_, f)
print("Saved svm_model.joblib to", MODEL_DIR)


## Step 7b — Model 2: XGBoost (Gradient Boosting) with early stopping

Early stopping on the held-out **validation** set (not test) prevents the
booster from over-fitting the training folds — a key guard against
overfitting for tree ensembles.

In [ ]:
xgb_model = xgb.XGBClassifier(
    n_estimators=600,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    objective="multi:softprob",
    num_class=NUM_CLASSES,
    eval_metric="mlogloss",
    early_stopping_rounds=30,
    random_state=SEED,
    n_jobs=-1,
)
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=False,
)
print("Best iteration:", xgb_model.best_iteration)

# Save immediately so this model is safe on Drive even if a later cell fails
xgb_model.save_model(os.path.join(MODEL_DIR, "xgb_model.json"))
print("Saved xgb_model.json to", MODEL_DIR)


In [ ]:
# Training-vs-validation loss curves -> visual overfitting check for XGBoost
results = xgb_model.evals_result()
plt.figure(figsize=(7,4))
plt.plot(results["validation_0"]["mlogloss"], label="train")
plt.plot(results["validation_1"]["mlogloss"], label="validation")
plt.axvline(xgb_model.best_iteration, color="grey", linestyle="--", label="best iteration")
plt.xlabel("boosting round"); plt.ylabel("mlogloss"); plt.legend()
plt.title("XGBoost: train vs validation loss")
plt.show()


## Step 7c — Model 3: CNN on Mel-Spectrograms (Deep Learning)

Built with `tf.data` so augmented + original training clips are streamed
from disk (no need to hold all spectrograms in RAM). Overfitting is
controlled with: dropout, batch normalization, L2 weight decay, early
stopping, and learning-rate reduction on plateau.

In [ ]:
def make_mel_dataset(subset_df, batch_size=32, shuffle=False, cache_path=None):
    if cache_path and os.path.exists(cache_path):
        data = np.load(cache_path, allow_pickle=True)
        X, y = data["X"], data["y"]
    else:
        X, y = [], []
        for _, row in tqdm(subset_df.iterrows(), total=len(subset_df), desc="Mel images"):
            yy = load_audio(row.filepath)
            X.append(extract_mel_image(yy))
            y.append(row.label)
        X = np.stack(X)[..., np.newaxis]   # add channel dim
        y = label_encoder.transform(y)
        if cache_path:
            np.savez_compressed(cache_path, X=X, y=y)
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(X), seed=SEED)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds, X.shape[1:]

train_ds, mel_shape = make_mel_dataset(train_full_df, batch_size=32, shuffle=True,
                                        cache_path=os.path.join(WORK_DIR, "mel_train.npz"))
val_ds, _   = make_mel_dataset(val_df, batch_size=32, shuffle=False,
                                cache_path=os.path.join(WORK_DIR, "mel_val.npz"))
test_ds, _  = make_mel_dataset(test_df, batch_size=32, shuffle=False,
                                cache_path=os.path.join(WORK_DIR, "mel_test.npz"))
print("Mel image shape:", mel_shape)


In [ ]:
def build_cnn(input_shape, num_classes, l2=1e-4):
    reg = regularizers.l2(l2)
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv2D(32, 3, padding="same", kernel_regularizer=reg)(inputs)
    x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv2D(64, 3, padding="same", kernel_regularizer=reg)(x)
    x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Conv2D(128, 3, padding="same", kernel_regularizer=reg)(x)
    x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)

    x = layers.Dense(128, kernel_regularizer=reg)(x)
    x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = models.Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model

cnn_model = build_cnn(mel_shape, NUM_CLASSES)
cnn_model.summary()


In [ ]:
cnn_callbacks = [
    callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6),
    callbacks.ModelCheckpoint(os.path.join(WORK_DIR, "best_cnn.keras"),
                               monitor="val_loss", save_best_only=True),
]

history = cnn_model.fit(
    train_ds, validation_data=val_ds,
    epochs=60, callbacks=cnn_callbacks, verbose=1
)

# Save immediately so this model is safe on Drive even if a later cell fails.
# EarlyStopping(restore_best_weights=True) means cnn_model already holds the
# best-validation-loss weights at this point.
cnn_model.save(os.path.join(MODEL_DIR, "cnn_model.keras"))
print("Saved cnn_model.keras to", MODEL_DIR)


In [ ]:
# Training-vs-validation curves -> visual overfitting check for the CNN
fig, ax = plt.subplots(1, 2, figsize=(12,4))
ax[0].plot(history.history["loss"], label="train"); ax[0].plot(history.history["val_loss"], label="val")
ax[0].set_title("Loss"); ax[0].legend()
ax[1].plot(history.history["accuracy"], label="train"); ax[1].plot(history.history["val_accuracy"], label="val")
ax[1].set_title("Accuracy"); ax[1].legend()
plt.tight_layout(); plt.show()

train_val_gap = history.history["accuracy"][-1] - history.history["val_accuracy"][-1]
print(f"Final train-val accuracy gap: {train_val_gap:.3f} (rule of thumb: flag if > 0.10)")


## Step 8 — Unified Evaluation on the Held-Out TEST Set

Computes accuracy, precision, recall, F1, macro-F1, confusion matrix,
class-wise performance, and critical-event recall for all three models.

In [ ]:
def evaluate_predictions(y_true, y_pred, model_name, class_names=classes):
    """Computes accuracy, per-class precision/recall/F1, macro- and
    weighted-averaged precision/recall/F1, a confusion matrix, and
    critical-event recall. Returns a flat dict of scalar metrics (used for
    the cross-model comparison) plus the class-wise breakdown."""
    acc = accuracy_score(y_true, y_pred)

    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, average=None, labels=range(len(class_names))
    )
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro"
    )
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted"
    )

    print(f"\n===== {model_name} =====")
    print(f"Accuracy:            {acc:.4f}")
    print(f"Macro Precision:     {precision_macro:.4f}")
    print(f"Macro Recall:        {recall_macro:.4f}")
    print(f"Macro F1:            {f1_macro:.4f}")
    print(f"Weighted Precision:  {precision_weighted:.4f}")
    print(f"Weighted Recall:     {recall_weighted:.4f}")
    print(f"Weighted F1:         {f1_weighted:.4f}")
    print("\nClassification report:\n", classification_report(y_true, y_pred, target_names=class_names, digits=3))

    cw = pd.DataFrame({
        "class": class_names, "precision": precision, "recall": recall,
        "f1": f1, "support": support
    })
    print("Class-wise performance:\n", cw.to_string(index=False))

    crit_recall = np.nan
    if CRITICAL_CLASSES:
        crit_idx = [class_names.index(c) for c in CRITICAL_CLASSES if c in class_names]
        if crit_idx:
            crit_recall = np.mean([recall[i] for i in crit_idx])
            print(f"\nCritical-event recall ({CRITICAL_CLASSES}): {crit_recall:.4f}")

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8,7))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.title(f"{model_name} — Confusion Matrix")
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.xticks(rotation=45, ha="right"); plt.tight_layout()
    plt.show()

    return {
        "model": model_name,
        "accuracy": acc,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted,
        "critical_event_recall": crit_recall,
        "class_wise": cw,
    }

results_summary = []

svm_pred = svm_model.predict(X_test)
results_summary.append(evaluate_predictions(y_test, svm_pred, "SVM"))

xgb_pred = xgb_model.predict(X_test)
results_summary.append(evaluate_predictions(y_test, xgb_pred, "XGBoost"))

cnn_probs = cnn_model.predict(test_ds)
cnn_pred = np.argmax(cnn_probs, axis=1)
results_summary.append(evaluate_predictions(y_test, cnn_pred, "CNN"))


In [ ]:
summary_df = pd.DataFrame([{k: v for k, v in r.items() if k != "class_wise"} for r in results_summary])
summary_df = summary_df.sort_values("f1_macro", ascending=False).reset_index(drop=True)
print(summary_df)

best_model_name = summary_df.iloc[0]["model"]
print(f"\nBest model by macro-F1: {best_model_name}")


## Step 9 — Noise-Robustness Evaluation

Re-runs the test set through each model after injecting background noise at
several SNR levels, to check how gracefully performance degrades — a proxy
for real-world robustness rather than clean-lab accuracy.

In [ ]:
def noisy_tabular_features(subset_df, snr_db):
    rows = []
    for _, row in subset_df.iterrows():
        y = load_audio(row.filepath)
        y_noisy = add_background_noise(y, snr_db=snr_db)
        f = extract_tabular_features(y_noisy)
        f["label"] = row.label
        rows.append(f)
    return pd.DataFrame(rows)

def noisy_mel_batch(subset_df, snr_db):
    X = []
    for _, row in subset_df.iterrows():
        y = load_audio(row.filepath)
        y_noisy = add_background_noise(y, snr_db=snr_db)
        X.append(extract_mel_image(y_noisy))
    return np.stack(X)[..., np.newaxis]

snr_levels = [20, 10, 5, 0]
robustness_rows = []

sample_df = test_df.sample(min(120, len(test_df)), random_state=SEED)  # subsample for speed; raise if you want the full test set

for snr in snr_levels:
    feat_noisy = noisy_tabular_features(sample_df, snr)
    Xn = scaler.transform(feat_noisy[feature_cols])
    yn = label_encoder.transform(feat_noisy.label)

    svm_acc = accuracy_score(yn, svm_model.predict(Xn))
    xgb_acc = accuracy_score(yn, xgb_model.predict(Xn))

    Xn_mel = noisy_mel_batch(sample_df, snr)
    cnn_acc = accuracy_score(yn, np.argmax(cnn_model.predict(Xn_mel), axis=1))

    robustness_rows.append({"SNR_dB": snr, "SVM_acc": svm_acc, "XGBoost_acc": xgb_acc, "CNN_acc": cnn_acc})

robust_df = pd.DataFrame(robustness_rows)
print(robust_df)

robust_df.set_index("SNR_dB")[["SVM_acc","XGBoost_acc","CNN_acc"]].plot(marker="o", figsize=(7,4))
plt.gca().invert_xaxis()
plt.xlabel("SNR (dB) — lower = noisier"); plt.ylabel("Accuracy")
plt.title("Noise-Robustness"); plt.grid(alpha=0.3)
plt.show()


## Step 10 — Overfitting Summary Check

Compares train vs. test performance for every model. A large gap flags
overfitting even when test accuracy looks acceptable in isolation.

In [ ]:
def train_test_gap(model_name, model, is_cnn=False):
    if is_cnn:
        train_probs = model.predict(train_ds)
        train_pred = np.argmax(train_probs, axis=1)
        train_true = np.concatenate([y.numpy() for _, y in train_ds.unbatch().batch(1024)])
    else:
        train_pred = model.predict(X_train)
        train_true = y_train
    train_acc = accuracy_score(train_true, train_pred)
    test_acc = summary_df.set_index("model").loc[model_name, "accuracy"]
    gap = train_acc - test_acc
    return {"model": model_name, "train_acc": train_acc, "test_acc": test_acc, "gap": gap}

gaps = [
    train_test_gap("SVM", svm_model),
    train_test_gap("XGBoost", xgb_model),
    train_test_gap("CNN", cnn_model, is_cnn=True),
]
gap_df = pd.DataFrame(gaps)
print(gap_df)
print("\nRule of thumb: a gap above ~0.10 (10 points) suggests overfitting —")
print("revisit regularization strength, augmentation volume, or model capacity for that model.")


## Step 11 — Save Remaining Artifacts (Preprocessing + Results)

The three trained models (SVM, XGBoost, CNN) were each already saved to
`MODEL_DIR` right after their own training cell above, so partial progress
survives even if a later cell errors out. This cell saves everything else
needed to reload and reuse them: the feature scaler, label encoder, the
tabular feature column list, and the evaluation/comparison results.

In [ ]:
joblib.dump(scaler,        os.path.join(MODEL_DIR, "feature_scaler.joblib"))
joblib.dump(label_encoder, os.path.join(MODEL_DIR, "label_encoder.joblib"))

with open(os.path.join(MODEL_DIR, "feature_cols.json"), "w") as f:
    json.dump(feature_cols, f)

summary_df.to_csv(os.path.join(MODEL_DIR, "model_comparison_summary.csv"), index=False)
for r in results_summary:
    r["class_wise"].to_csv(os.path.join(MODEL_DIR, f"classwise_{r['model']}.csv"), index=False)
gap_df.to_csv(os.path.join(MODEL_DIR, "train_test_gap.csv"), index=False)

confirm = {
    "svm_model.joblib":    os.path.exists(os.path.join(MODEL_DIR, "svm_model.joblib")),
    "xgb_model.json":      os.path.exists(os.path.join(MODEL_DIR, "xgb_model.json")),
    "cnn_model.keras":     os.path.exists(os.path.join(MODEL_DIR, "cnn_model.keras")),
    "feature_scaler.joblib": os.path.exists(os.path.join(MODEL_DIR, "feature_scaler.joblib")),
    "label_encoder.joblib":  os.path.exists(os.path.join(MODEL_DIR, "label_encoder.joblib")),
}
print("Artifact presence check:")
for name, ok in confirm.items():
    print(f"  [{'OK' if ok else 'MISSING'}] {name}")
assert all(confirm.values()), "One or more model/artifact files are missing — re-run the corresponding training cell."

print("\nAll 3 trained models + preprocessing artifacts + results saved to:", MODEL_DIR)
print(os.listdir(MODEL_DIR))


## Step 12 — Inference: Classify a New / Live Audio Clip

Returns a confidence score for every one of the 10 classes, using the
best-performing model from Step 8 (falls back to whichever you name
explicitly).

In [ ]:
def classify_audio(filepath, model_choice=None):
    """model_choice: 'SVM', 'XGBoost', 'CNN', or None (uses the best model from Step 8)."""
    model_choice = model_choice or best_model_name
    y = load_audio(filepath)

    if model_choice == "CNN":
        mel = extract_mel_image(y)[np.newaxis, ..., np.newaxis]
        probs = cnn_model.predict(mel, verbose=0)[0]
    else:
        feats = extract_tabular_features(y)
        X = scaler.transform(pd.DataFrame([feats])[feature_cols])
        if model_choice == "SVM":
            probs = svm_model.predict_proba(X)[0]
        elif model_choice == "XGBoost":
            probs = xgb_model.predict_proba(X)[0]
        else:
            raise ValueError("model_choice must be 'SVM', 'XGBoost', or 'CNN'")

    order = np.argsort(probs)[::-1]
    result = {classes[i]: float(probs[i]) for i in order}
    predicted_class = classes[order[0]]
    return predicted_class, result

# Example usage:
# pred_class, confidences = classify_audio("/content/drive/MyDrive/some_clip.wav")
# print("Predicted:", pred_class)
# for cls, score in confidences.items():
#     print(f"  {cls}: {score:.3f}")


## Step 13 — Final Comparison of All Three Models

Side-by-side table and chart of Accuracy, macro Precision, macro Recall,
macro F1, and weighted F1 for SVM, XGBoost, and the CNN on the same held-out
test set, plus the train/test overfitting gap from Step 10 for context.

In [ ]:
comparison_df = summary_df[[
    "model", "accuracy", "precision_macro", "recall_macro", "f1_macro", "f1_weighted"
]].rename(columns={
    "model": "Model", "accuracy": "Accuracy", "precision_macro": "Precision (macro)",
    "recall_macro": "Recall (macro)", "f1_macro": "F1 (macro)", "f1_weighted": "F1 (weighted)"
}).set_index("Model")

comparison_df = comparison_df.join(gap_df.set_index("model")["gap"].rename("Train-Test Gap"))

print("Final model comparison (test set):\n")
display(comparison_df.round(4))

ax = comparison_df[["Accuracy", "Precision (macro)", "Recall (macro)", "F1 (macro)"]].plot(
    kind="bar", figsize=(9, 5), ylim=(0, 1)
)
ax.set_title("Model Comparison — Test Set")
ax.set_ylabel("Score")
ax.legend(loc="lower right")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print(f"\nBest overall model (by macro F1): {best_model_name}")


### Notes / knobs worth tuning

- **`AUGS_PER_CLIP`** — raise for more augmentation diversity; watch training
  time and disk usage grow linearly.
- **`CRITICAL_CLASSES`** — fill this in with the class name(s) that matter
  most for missed-detection cost (e.g. alarms/sirens); the evaluation cell
  reports their recall specifically.
- **Overfitting levers already in place**: stratified CV for SVM,
  early stopping + L1/L2 regularization for XGBoost, dropout + batch norm +
  L2 + early stopping + LR scheduling for the CNN, plus augmentation applied
  only to the training split so validation/test stay a clean, honest
  estimate of generalization.
- If the CNN overfits despite all of this, first try: fewer epochs before
  early-stopping patience, stronger dropout, or increasing `AUGS_PER_CLIP`
  before adding model capacity.
